In [1]:
from loguru import logger
from transformers import BartForConditionalGeneration, BertForSequenceClassification
from os.path import join

from transformers import BartConfig, BertConfig
from carl.inference_components.subgoal_generator import AdaptiveSubgoalGenerator
from carl.environment.n_puzzle.env import NPuzzleEnv
from carl.environment.n_puzzle.tokenizer import NPuzzleTokenizer
from carl.inference_components.validator import BasicValidator
from carl.inference_components.conditional_low_level_policy import (
    ConditionalLowLevelPolicy,
    TransformerConditionalLowLevelPolicy,
)
from carl.inference_components.subgoal_generator import (
    TransformerSubgoalGenerator,
)
from carl.inference_components.value import TransformerValue, Value
import functools

logger.info('testing solve components')

components_prefix = './rl-data/validation/npuzzle/components/moe/'

path_to_cllp_weights: str = join(components_prefix, 'cllp/4/checkpoint-294075')
path_to_value_function_weights: str = join(components_prefix, 'value/checkpoint-2825298')
path_to_generator_weights_k4: str = join(components_prefix, 'generator/4/checkpoint-48314')

tokenizer = NPuzzleTokenizer()
env_class = functools.partial(NPuzzleEnv, tokenizer=tokenizer)
env = env_class()

subgoal_generation_kwargs: dict[str, int] = {
    'max_new_tokens': 26,
    'num_beams': 4,
    'do_sample': False,
    'temperature': 1.0,
    'num_return_sequences': 1,
}

subgoal_generator_cls = functools.partial(
    TransformerSubgoalGenerator, generator_network_class=BartForConditionalGeneration.from_pretrained
)

adaptive_subgoal_generator = AdaptiveSubgoalGenerator(
    generator_k_list=[4],
    subgoal_generator_class=subgoal_generator_cls,
    paths_to_generator_weights=[path_to_generator_weights_k4],
    env=env,
    subgoal_generation_kwargs=subgoal_generation_kwargs,
)

adaptive_subgoal_generator.construct_network()


value_function: Value = TransformerValue(
    value_network_class=BertForSequenceClassification.from_pretrained,
    path_to_value_network_weights=path_to_value_function_weights,
    env=env,
    type_of_evaluation='regression',
)
value_function.construct_network()

cllp: ConditionalLowLevelPolicy = TransformerConditionalLowLevelPolicy(
    BertForSequenceClassification.from_pretrained, path_to_cllp_weights, env
)
cllp.construct_network()

validator = BasicValidator(env_class(), cllp, budget_for_achieving_subgoal=8)
validator.construct_network()

2025-01-28 03:51:11.500 | INFO     | __main__:<module>:28 - testing solve components
2025-01-28 03:51:11.502 | DEBUG    | carl.inference_components.component:instantiate_network:52 - Loading weights from ./rl-data/validation/npuzzle/components/moe/generator/4/checkpoint-48314
2025-01-28 03:51:11.503 | INFO     | carl.inference_components.component:instantiate_network:73 - Provided direct path to the ckpt
2025-01-28 03:51:11.542 | SUCCESS  | carl.inference_components.component:instantiate_network:85 - Loaded weights from ./rl-data/validation/npuzzle/components/moe/generator/4/checkpoint-48314
2025-01-28 03:51:14.099 | DEBUG    | carl.inference_components.component:instantiate_network:52 - Loading weights from ./rl-data/validation/npuzzle/components/moe/value/checkpoint-2825298
2025-01-28 03:51:14.101 | INFO     | carl.inference_components.component:instantiate_network:73 - Provided direct path to the ckpt
2025-01-28 03:51:14.116 | SUCCESS  | carl.inference_components.component:instantia

In [2]:
import functools
from carl.solver.planners import AdasubsPlanner
from carl.solver.subgoal_search import Solver

AdasubsPlannerCls = functools.partial(AdasubsPlanner, generators_k_list=[4])

solver = Solver(
    500,
    AdasubsPlannerCls,
    adaptive_subgoal_generator,
    validator,
    value_function,
)

from carl.environment.instance_generator import (
    BasicInstanceGenerator,
    GeneralIterableDataLoader,
)

path_to_folder_with_data = './rl-data/validation/npuzzle/progress/fin'

instance_generator = BasicInstanceGenerator(
    generator=GeneralIterableDataLoader(path_to_folder_with_data), batch_size=1
)

initial_state_loader = iter(instance_generator.reset_dataloader())
inputs = []
figs = []
for _ in range(3):
    initial_state = next(initial_state_loader).cpu().numpy()[0]
    inputs.append(initial_state)
    
fig = env.many_states_to_repr(inputs, titles=['input_{}'.format(i) for i in range(len(inputs))])
print(fig)

2025-01-28 03:51:14.965 | DEBUG    | carl.environment.instance_generator:__init__:36 - path: ./rl-data/validation/npuzzle/progress/fin
2025-01-28 03:51:14.968 | DEBUG    | carl.environment.instance_generator:get_stream:55 - Found 1 files in ./rl-data/validation/npuzzle/progress/fin
2025-01-28 03:51:14.969 | INFO     | carl.environment.instance_generator:process_data:45 - Processing data from ./rl-data/validation/npuzzle/progress/fin/fin_puzzles_1000.pkl


input_0: [20 23  0  3  1  2 13 19  5 15 21  9 11  4 16  7 18 14 12  8 10 22 24  6
 17]
input_1: [ 1 16  4 20  9  2 11  0  8 21 22 24 18 12 13 17  6  7 15 14  3 19 10  5
 23]
input_2: [ 1  8 19 13  6  3  7 23  9 14  2 16  5 15 20 10 11 21  4 24 17  0 18 22
 12]


In [3]:
outputs_sequential = []
for i, input_board in enumerate(inputs):
    print('Solving board number', i)
    output = solver.solve(input_board)
    outputs_sequential.append(output)
    
def display_solutions(outputs):
    figs = []
    for j, output in enumerate(outputs):
        solution, _ = output
        if solution['solved']:
            subgoals = solution['subgoal_path']
            titles = [
                f'subgoal[{i}]'
                for i in range(len(subgoals))
            ]
            
            fig = env.many_states_to_repr(subgoals, titles=titles)
        else:
            fig = env.state_to_repr(initial_state, title='initial_state (unsolved)')

        figs.append(fig)

    for fig in figs:
        print(fig)

Solving board number 0
Rising recursion limit for worker from 3000 to 2147483640.


2025-01-28 03:51:52.231 | SUCCESS  | carl.solver.subgoal_search:solve:110 - Solved.


Solving board number 1


2025-01-28 03:52:17.556 | SUCCESS  | carl.solver.subgoal_search:solve:110 - Solved.


Solving board number 2


2025-01-28 03:52:37.199 | SUCCESS  | carl.solver.subgoal_search:solve:110 - Solved.
